# Temporal Gesture Classifier — GRU-64 Training

Train a **GRU-64** model on sequence data collected via `SequenceCollector`.

- **Input**: NPZ file with `sequences` (N, 20, 93) and `labels` (N,)
- **Output**: TFLite model at `model/temporal_classifier/temporal_classifier.tflite`
- **Architecture**: Input(20, 93) → GRU(64, unroll=True) → Dropout(0.3) → Dense(5, softmax)
- **Design doc**: `docs/ai/design/feature-temporal-gru64.md`

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [ ]:
# --- Configuration ---
NPZ_PATH = "model/temporal_classifier/keypoint_sequences.npz"
KERAS_SAVE_PATH = "model/temporal_classifier/temporal_classifier.keras"
TFLITE_SAVE_PATH = "model/temporal_classifier/temporal_classifier.tflite"

WINDOW_SIZE = 20
NUM_FEATURES = 93
NUM_CLASSES = 5
HIDDEN_UNITS = 64

CLASS_NAMES = ["null", "pointer_move", "left_click", "drag_hold", "scroll_mode"]

## Load & Explore Data

In [ ]:
# --- Load sequence data ---
data = np.load(NPZ_PATH)
sequences = data["sequences"]  # (N, 20, 93)
labels = data["labels"]        # (N,)

print(f"Sequences shape: {sequences.shape}")
print(f"Labels shape:    {labels.shape}")
print(f"Unique classes:  {np.unique(labels)}")
print(f"dtype:           {sequences.dtype}")

In [ ]:
# --- Class distribution ---
unique, counts = np.unique(labels, return_counts=True)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar([CLASS_NAMES[i] for i in unique], counts, color="steelblue")
ax.bar_label(bars)
ax.set_title("Sequence count per class")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## Train / Validation Split

In [ ]:
# --- Train / Validation split (75/25, stratified) ---
X_train, X_val, y_train, y_val = train_test_split(
    sequences, labels, test_size=0.25, random_state=42, stratify=labels
)
print(f"Train: {X_train.shape[0]}  |  Val: {X_val.shape[0]}")

## Build & Train GRU-64 Model

In [ ]:
# --- Build GRU-64 model ---
model = tf.keras.Sequential([
    tf.keras.layers.Input((WINDOW_SIZE, NUM_FEATURES)),
    tf.keras.layers.GRU(HIDDEN_UNITS, unroll=True, return_sequences=False),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(NUM_CLASSES, activation="softmax"),
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

In [ ]:
# --- Train ---
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        KERAS_SAVE_PATH, save_best_only=True, monitor="val_accuracy"
    ),
    tf.keras.callbacks.EarlyStopping(
        patience=20, restore_best_weights=True, monitor="val_accuracy"
    ),
]

history = model.fit(
    X_train, y_train,
    epochs=300,
    batch_size=128,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
)

## Evaluate

In [ ]:
# --- Training curves ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history["loss"], label="train")
ax1.plot(history.history["val_loss"], label="val")
ax1.set_title("Loss")
ax1.set_xlabel("Epoch")
ax1.legend()

ax2.plot(history.history["accuracy"], label="train")
ax2.plot(history.history["val_accuracy"], label="val")
ax2.set_title("Accuracy")
ax2.set_xlabel("Epoch")
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# --- Classification report + Confusion matrix ---
y_pred = np.argmax(model.predict(X_val), axis=1)
print(classification_report(y_val, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_val, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax,
)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix")
plt.tight_layout()
plt.show()

## Export to TFLite

In [ ]:
# --- Convert to TFLite (dynamic quantization) ---
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open(TFLITE_SAVE_PATH, "wb") as f:
    f.write(tflite_model)

size_kb = os.path.getsize(TFLITE_SAVE_PATH) / 1024
print(f"TFLite model saved: {TFLITE_SAVE_PATH} ({size_kb:.1f} KB)")

In [ ]:
# --- Verify TFLite inference ---
interpreter = tf.lite.Interpreter(model_path=TFLITE_SAVE_PATH)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(f"Input:  {input_details[0]['shape']}")
print(f"Output: {output_details[0]['shape']}")

# Compare Keras vs TFLite on a few samples
n_check = min(10, len(X_val))
keras_preds = np.argmax(model.predict(X_val[:n_check]), axis=1)
tflite_preds = []
for i in range(n_check):
    interpreter.set_tensor(
        input_details[0]["index"], X_val[i : i + 1].astype(np.float32)
    )
    interpreter.invoke()
    scores = interpreter.get_tensor(output_details[0]["index"])
    tflite_preds.append(int(np.argmax(scores)))

print(f"\nKeras  preds: {list(keras_preds)}")
print(f"TFLite preds: {tflite_preds}")
print(f"Match: {np.array_equal(keras_preds, tflite_preds)}")

**Done!** The trained TFLite model is at `model/temporal_classifier/temporal_classifier.tflite`.

Use `TemporalClassifier` wrapper to load it for inference:
```python
from model.temporal_classifier import TemporalClassifier
tc = TemporalClassifier()
class_id, scores = tc(window)  # window: (20, 93) ndarray
```